**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/russo-2025"], check=True)
    os.chdir("/content/encore/entries/russo-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Russo, Claar, Furregoni et al. (2025) from dandiset 000458

Paper. Russo S, Claar LD, Furregoni G, Marks LC, Krishnan G, Zauli FM, et al., Koch C, Rembado I
(2025). Thalamic feedback shapes brain responses evoked by cortical stimulation in mice and
humans. *Nature Communications* 16:3627.
[doi:10.1038/s41467-025-58717-2](https://doi.org/10.1038/s41467-025-58717-2). The claims file
(`claims/russo-2025.yaml`) was written from the version of record in PubMed Central, with the
supplement and the Source Data file.

Dandiset. [DANDI:000458](https://dandiarchive.org/dandiset/000458/0.230317.0039), version
0.230317.0039 (Claar et al. 2023, "Simultaneous electroencephalography, extracellular
electrophysiology, and cortical electrical stimulation in head-fixed mice"). This is the version
the paper's Data availability statement cites, and the only published version. The draft has the
same 24 assets and size and lists 10 active uploads; it was not used.

What the dandiset holds. 24 sessions from 23 mice, each with 30-channel EEG, the LFP of up to
three Neuropixels probes, sorted units, wheel speed and a trial table, in an awake, an isoflurane
and sometimes a recovery epoch. Ten sessions (ten mice) have deep MOs stimulation while awake, nine
of them with probes; these are the mouse data used here. The paper reports 26 EEG sessions from 15
mice and units from 21 mice, so the dandiset holds part of them. No session has stimulation of
the posterior control areas of Figure S6.

Outside inputs. The human claims are tested on the paper's Source Data file
(41467_2025_58717_MOESM4_ESM.xlsx from the publisher's site, checked by SHA-256), which is also the
reference for the link claim. The optogenetic claims are in the second notebook,
`opto_russo_2025.ipynb`, on the paper's figshare deposit (doi:10.6084/m9.figshare.28440311.v1).

Running. The helper `russo.py` streams the needed chunks of each file from the archive in
parallel; the notebook takes about 15 minutes, nearly all of it reading the EEG and the MOs LFP of
the ten sessions.
The model section fetches and builds the paper's network model, which needs `make` and `g++`.

| Claim | Tested here | Outcome |
|---|---|---|
| source-data-link | yes | partially reproduced |
| fig1a-tms-late-rest-active | Source Data | reproduced |
| fig1b-ieeg-late-rest-active | Source Data | reproduced |
| figs2-tms-controls | Source Data | reproduced |
| figs2a-tms-early-component | Source Data | reproduced |
| figs3b-ieeg-early-no-modulation | Source Data | reproduced |
| fig5a-human-mouse-similar | no | blocked, data missing |
| fig1c-eeg-late-rest-larger | yes | reproduced |
| fig1d-lfp-late-rest-larger | yes | reproduced |
| fig1e-csd-late-rest-larger | yes | reproduced |
| figs1b-mouse-plf | yes | reproduced |
| figs4-intensity-dependence | yes | reproduced |
| speed-threshold-robust | yes | not reproduced |
| figs5-early-component-mouse | yes | reproduced |
| figs6-posterior-no-modulation | no | blocked, data missing |
| triphasic-pattern | yes | partially reproduced |
| fig2b-thalamus-leads | yes | not reproduced |
| figs7a-active-shorter-latency | yes | reproduced |
| figs7b-latency-correlation | yes | reproduced |
| fig3a-thalamic-rate-higher-active | yes | reproduced |
| figs9-mos-rate-no-modulation | yes | not reproduced |
| figs10a-fs-trn-no-modulation | yes | reproduced (fragile) |
| fig3b-bimodal-isi | yes | not reproduced |
| figs11d-hf-earlier-less-variable | yes | not reproduced |
| figs12-intensity-shifts-to-hf | yes | reproduced |
| fig3e-baseline-rate-doubles | yes | reproduced |
| fig3e-lf-synchrony-reduced | yes | not reproduced |
| fig3e-hf-frequency-reduced | yes | reproduced |
| figs13-thalamic-features-predict-eeg | yes | not reproduced |
| figs13d-lf-crosscorrelation | yes | not reproduced |
| discussion-orthodromic-initial-excitation | yes | reproduced |
| fig2fg, figs8b, fig2h (optogenetics) | second notebook | see there |
| fig4e, fig4f, fig4hi, figs14a, figs14b (model) | build and timing only | blocked, not runnable (too slow for the time limit) |

In [ ]:
import hashlib
import io
import time
import warnings
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.cluster import KMeans
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

import diptest
import russo

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)


def claim_seed(claim):
    """Seed of a claim's generator (docs/protocol.md, Author, step 3)."""
    return zlib.crc32(f"russo-2025/{claim}".encode())


started = time.time()
index = russo.session_index()
deep = index[(index.region == "MOs") & (index.depth == "deep")]
S = {p: russo.process_session(p) for p in deep.path}
print(f"dandiset {russo.DANDISET_ID} version {russo.DANDISET_VERSION}: {index.path.nunique()} files; "
      f"{len(S)} sessions with deep MOs stimulation loaded in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Awake electrical stimulation blocks in every file (target region and depth, currents in uA):")
print(index.to_string(index=False))
print("\nTarget regions in the trial tables:", sorted(index.region.unique()), "; depths:", sorted(index.depth.unique()))
print("Subjects in the dandiset:", index.subject.nunique(), "; files:", index.path.nunique())
print("Sessions with deep MOs stimulation:", len(deep), "from", deep.subject.nunique(), "mice;",
      "with Neuropixels probes:", int((deep.n_units > 0).sum()))

The trial tables hold electrical stimulation of MOs (deep and superficial) and of SSp (deep and
superficial). No file has a stimulation target in visual or retrosplenial cortex or hippocampus,
the control sessions of Figure S6. The paper's mouse results use deep MOs stimulation (Results:
"targeting the deep layers of MOs (layer 5/6...)"); ten sessions from ten mice have it, nine of
them with Neuropixels probes and units. The paper reports 26 EEG sessions from 15 mice (Figure 1C)
and units from 21 mice (Figure S7B), so the dandiset holds part of its mouse data.

In [ ]:
rows = []
for p, r in S.items():
    t = r["trials"]
    cur = t.groupby("intensity").current.first()
    rows.append({"subject": r["subject"], "trials": len(t), "valid": int(t.is_valid.sum()),
                 "low_uA": cur.get("low"), "medium_uA": cur.get("medium"), "high_uA": cur.get("high"),
                 "rest_0.1": int((~t.active).sum()), "active_0.1": int(t.active.sum()),
                 "speed_min": round(t.speed.min(), 3), "speed_max": round(t.speed.max(), 1),
                 "file_is_running": int(t.is_running.sum()), "speed>0": int((t.speed > 0).sum()),
                 "agree_is_running_vs_speed>0": round(float((t.is_running == (t.speed > 0)).mean()), 3),
                 "eeg_valid_channels": r["n_eeg_valid"], "MOs_LFP_channels": r.get("n_mos_channels"),
                 "eeg_artifact_lag_ms": r["eeg_artifact_lag_ms"], "lfp_artifact_lag_ms": r.get("lfp_artifact_lag_ms"),
                 "eeg_onset_err_ms": round(r["eeg_onset_err_ms"], 3), "lfp_onset_err_ms": r.get("lfp_onset_err_ms")})
sessions = pd.DataFrame(rows)
print(sessions.to_string(index=False))
for k in ["low_uA", "medium_uA", "high_uA"]:
    print(f"{k}: mean {sessions[k].mean():.2f} +- {sessions[k].std():.2f} (paper: high 66.15 +- 12.11, "
          "intermediate 47.35 +- 13.30, low 30.59 +- 15.80)")
ranges = np.array([r["eeg_range_uv"] for r in S.values()])
print(f"EEG after preprocessing, -1 to +0.5 s epochs: {ranges[:, 0].min():.0f} to {ranges[:, 1].max():.0f} uV")

Every session has 120 trials at each of three currents, as the Methods state (up to 120 pulses per
intensity). The currents of this subset (printed above) are close to the paper's means. Wheel
speed is in cm/s and ranges from 0 to tens of cm/s. The file's `is_running` is defined as a mean
speed above 0 cm/s; the agreement column shows it matches the mean speed from -0.5 to +0.5 s
computed here from the `running_speed` series, which is consistent with the trial times and the
speed sharing a clock (both come from the same pipeline, so this does not prove it). The paper's
threshold is 0.1 cm/s, used below.

Clock of the signals: the onset of each trial falls within one sample (0.4 ms) of an EEG and LFP
timestamp, and the stimulation artifact in the raw trial-averaged EEG and LFP sits within one
sample of the onset in every session (columns `*_artifact_lag_ms`), so the EEG, the LFP and the
trial table share a clock.

In [ ]:
units = pd.concat([r["units"].assign(subject=r["subject"]) for r in S.values() if r.get("has_probes")], ignore_index=True)
print("waveform_duration over all units: min %.3f, median %.3f, max %.3f (the file says seconds)" %
      (units.waveform_duration.min(), units.waveform_duration.median(), units.waveform_duration.max()))
print("\nUnits by class and mouse (MOs RS > 0.4 ms, FS <= 0.4; SM-TH relay > 0.45, TRN < 0.35):")
print(pd.crosstab(units.subject, units.cls).to_string())
print("\nSM-TH nuclei of the units:", units[units.area == "SM-TH"].location.value_counts().to_dict())
print("Thalamic reticular nucleus (RT) units by location:", int((units.location == "RT").sum()))

U = pd.concat([russo.unit_trial_table(r) for r in S.values() if r.get("has_probes")], ignore_index=True)
U["active"] = U.speed > russo.SPEED_THRESHOLD
U = U[U.is_valid]
H = U[(U.intensity == "high") & U.responsive]
resp = H.groupby(["subject", "cls"]).unit.nunique().unstack(fill_value=0)
print("\nResponsive units at high intensity (7 SD rule), by mouse:")
print(resp.to_string())
print("Totals:", resp.sum().to_dict(), "(paper: 1269 relay units from 21 mice, 67 TRN, 145 FS)")

# Spike clock: population response of responsive MOs RS units in 2-10 ms against the same window
# of onsets shifted by +1.7 s (between pulses, 3.5-4.5 s apart).
rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    hi = (t.intensity == "high").to_numpy()
    on, sh = [], []
    for i, (rel, trial) in r["spikes"].items():
        if r["units"].loc[i, "cls"] not in ("RS", "relay"):
            continue
        sel = hi[trial]
        on.append(((rel[sel] >= 0.002) & (rel[sel] < 0.010)).sum() / hi.sum() / 0.008)
        sh.append(((rel[sel] >= -0.3) & (rel[sel] < -0.292)).sum() / hi.sum() / 0.008)
    rows.append({"subject": r["subject"], "units": len(on), "rate_2_10ms": np.mean(on), "rate_-300_-292ms": np.mean(sh)})
print("\nMean rate of RS and relay units in 2-10 ms after the pulse and in an 8 ms window 300 ms before it (Hz):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

`waveform_duration` is described in the files as seconds; its values (0.110 to 2.074, cell above) are
milliseconds, and are read as such. The spiking rate in the 2 to 10 ms after the pulse is far
above the rate in a window of equal length before it in every mouse, so the units and the trial
times share a clock. Unit analyses use the trials the file marks valid (`is_valid`); the paper
removed bad trials using a lab spreadsheet that is not public.

In [ ]:
def state_table(sig, intens=("high",), thr=0.1, chan="all", win="late", meas="amp"):
    """One row per session and intensity: the measure at rest and during running."""
    rows = []
    for r in S.values():
        if sig not in r:
            continue
        d = r[sig]
        d = d[(d.threshold == thr) & (d.chan == chan) & (d.window == win) & d.intensity.isin(intens)]
        for i, g in d.groupby("intensity"):
            g = g.set_index("state")
            rows.append({"subject": r["subject"], "intensity": i, "rest": g.loc["rest", meas],
                         "active": g.loc["active", meas], "n_rest": int(g.loc["rest", "n"]),
                         "n_active": int(g.loc["active", "n"])})
    return pd.DataFrame(rows)


def signed_rank(t, label, by_mouse=False):
    t = t.dropna(subset=["rest", "active"])
    if by_mouse:
        t = t.groupby("subject")[["rest", "active"]].mean().reset_index()
    w = stats.wilcoxon(t.rest, t.active) if len(t) > 1 else None
    return {"reading": label, "n": len(t), "rest_median": t.rest.median(), "active_median": t.active.median(),
            "rest_larger": int((t.rest > t.active).sum()), "p": (w.pvalue if w else np.nan),
            "effect_median_diff": (t.rest - t.active).median()}


READINGS = {"(a)(i) high, all channels and trials": (("high",), "all"),
            "(a)(ii) high, valid channels and trials": (("high",), "valid"),
            "(b)(i) high and medium, all": (("high", "medium"), "all"),
            "(b)(ii) high and medium, valid": (("high", "medium"), "valid")}


def readings_table(sig, meas="amp", win="late", thr=0.1, by_mouse_b=False):
    out = []
    for label, (intens, chan) in READINGS.items():
        t = state_table(sig, intens, thr, chan, win, meas)
        out.append(signed_rank(t, label))
        if by_mouse_b and len(intens) == 2:
            out.append(signed_rank(t, label + ", mean per mouse", by_mouse=True))
    return pd.DataFrame(out)

## fig1c-eeg-late-rest-larger

> In mice, the late (150 to 250 ms) EEG response to deep MOs electrical stimulation has a larger
> rectified amplitude at rest than during running.

Rectified amplitude: the absolute value of the trial-averaged EP from 150 to 250 ms, averaged over
time and EEG channels; two-tailed Wilcoxon signed-rank test pairing rest and active per session
(criterion). Readings as the criterion lists them: (a) high intensity only, (b) high and medium
as separate sessions; (i) all channels and trials, (ii) the channels (`is_data_valid`) and trials
(`is_valid`) the file flags as good. The code's measure (mean absolute single-trial signal after
baseline z-scoring) is a variant.

In [ ]:
res_1c = readings_table("eeg")
print(res_1c.round(6).to_string(index=False))
var = pd.DataFrame([signed_rank(state_table("eeg", intens, chan=chan, meas="code_amp"), "code measure " + lab)
                    for lab, (intens, chan) in READINGS.items()])
print("\nVariant, the code's measure (z units):")
print(var.round(6).to_string(index=False))
t1c = state_table("eeg")
print("\nPer session, reading (a)(i), uV:")
print(t1c.round(2).to_string(index=False))
print("Range of plotted amplitudes: %.1f to %.1f uV" % (t1c[["rest", "active"]].min().min(), t1c[["rest", "active"]].max().max()))

In [ ]:
tt = russo.times()
tw = tt[(tt >= -0.1) & (tt < 0.4)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for st, col in [("rest", "k"), ("active", "r")]:
    tr = np.mean([r["eeg_trace"][("high", st)] for r in S.values()], 0)
    ax[0].plot(tw * 1e3, tr, col, label=st)
ax[0].axvspan(150, 250, color="0.9")
ax[0].set_xlabel("ms from pulse"); ax[0].set_ylabel("|mean EP|, mean over channels (uV)"); ax[0].legend()
ax[1].plot([0, 1], t1c[["rest", "active"]].T.values, "o-", color="0.5")
ax[1].set_xticks([0, 1], ["rest", "active"]); ax[1].set_ylabel("late rectified amplitude (uV)")
plt.tight_layout(); plt.show()
traces = np.array([[r["eeg_trace"][("high", st)] for r in S.values()] for st in ["rest", "active"]]).mean(1)
print("Left panel range: %.1f to %.1f uV over -100 to 400 ms" % (traces.min(), traces.max()))

**Result: reproduced. The late EEG rectified amplitude is larger at rest than during running in
all 10 sessions at high intensity (median 81.7 against 42.6 uV, two-tailed signed-rank p =
0.00195), and the test gives p < 0.05 with rest larger under all four readings (p = 0.00195 for
both channel and trial selections at high intensity, p = 4e-06 with high and medium as 20
sessions). The code's z-scored single-trial measure agrees (p = 0.0039 and 4e-06). Ten sessions
from ten mice are used, where the paper reports 26 sessions from 15 mice.

## fig1d-lfp-late-rest-larger

> In mice, the late (150 to 250 ms) MOs LFP response to MOs stimulation has a larger rectified
> amplitude at rest than during running.

LFP of the Neuropixels channels the file places in MOs, detrended -2 to +2 s epochs, 0 to 3 ms
artifact mirrored, z-scored per channel and trial to -1 to -0.01 s; rectified amplitude of the
trial average over 150 to 250 ms and MOs channels (criterion). Each mouse has one session, so for
reading (a) sessions are mice; for reading (b) the test is shown across the 18 session and
intensity values and with the two intensities averaged per mouse.

In [ ]:
res_1d = readings_table("lfp", by_mouse_b=True)
print(res_1d.round(6).to_string(index=False))
t1d = state_table("lfp")
print("\nPer mouse, reading (a)(i), z units:")
print(t1d.round(3).to_string(index=False))

Result: reproduced. The late MOs LFP rectified amplitude (z units) is larger at rest in 7 of 9
mice at high intensity (median 1.98 against 1.54, p = 0.0195; with the valid channels and trials
p = 0.027), and in 16 of 18 session and intensity values with high and medium (p = 3.8e-05; 9 of 9
mice when the two are averaged, p = 0.0039).

## fig1e-csd-late-rest-larger

> In mice, the late MOs CSD response to MOs stimulation is larger at rest than during running,
> with a current sink between layers 2/3 and 5 at the time of the late response for high and
> medium but not low intensity.

CSD as in the Methods (helper docstring), z-scored like the LFP, over the MOs depth range. The
sink is described from the depth profile and does not decide the claim.

In [ ]:
res_1e = readings_table("csd", by_mouse_b=True)
print(res_1e.round(6).to_string(index=False))
t1e = state_table("csd")
print("\nPer mouse, reading (a)(i), z units:")
print(t1e.round(3).to_string(index=False))
# Depth profile at the late response: mean CSD (z) over 150-250 ms, by depth third of the MOs span, rest
tw = tt[(tt >= -0.1) & (tt < 0.4)]
lw = (tw >= 0.15) & (tw < 0.25)
rows = []
for r in S.values():
    if "csd_depth_profile" not in r:
        continue
    for inten in ["low", "medium", "high"]:
        prof = r["csd_depth_profile"][(inten, "rest")][:, lw].mean(1)
        thirds = np.array_split(prof[::-1], 3)  # from the surface (largest vertical position) down
        rows.append({"subject": r["subject"], "intensity": inten, **{f"third_{k + 1}": float(th.mean()) for k, th in enumerate(thirds)}})
prof = pd.DataFrame(rows)
print("\nMean late CSD at rest by third of the MOs depth range (1 = most superficial), median over mice:")
print(prof.groupby("intensity")[["third_1", "third_2", "third_3"]].median().round(3).to_string())

Result: reproduced. The late MOs CSD amplitude is larger at rest in 7 of 9 mice at high
intensity (median 1.26 against 0.76, p = 0.0195) and in 15 of 18 values with high and medium
(p = 0.00011). At rest the middle third of the MOs depth range carries a negative mean CSD (a
sink) in the late window at high (-0.69) and medium (-0.59) intensity and much less at low
(-0.09); this is a description and does not decide the claim.

## figs1b-mouse-plf

> In mice the PLF of the late response is larger at rest than during running for LFP and CSD,
> while for EEG the difference does not reach significance (p = 0.055).

PLF per channel is the absolute value of the trial average of the unit phase vectors of the
Hilbert transform, averaged over 150 to 250 ms and over channels (criterion), with the readings of
the three amplitude claims.

In [ ]:
for sig in ["lfp", "csd", "eeg"]:
    print(sig.upper())
    print(readings_table(sig, meas="plf").round(6).to_string(index=False))

Result: reproduced. The late PLF is larger at rest for the LFP (7 of 9 mice, p = 0.0195 at
high intensity; p = 0.00042 with high and medium) and the CSD (8 of 9, p = 0.027 and 0.020 for
the two selections; p = 0.0034 with high and medium). For the EEG, which does not decide the
claim, the difference is significant here (9 of 10 sessions, p = 0.0059), where the paper
reports p = 0.055.

## figs4-intensity-dependence

> The state modulation of the late response depends on stimulation intensity; EEG late amplitude
> is larger at rest for high and intermediate but not low intensity, LFP at all three, and CSD for
> high and intermediate but not low.

One value per mouse, state and intensity; all channels and trials, and the file's valid channels
and trials as a second reading.

In [ ]:
rows = []
for sig in ["eeg", "lfp", "csd"]:
    for chan in ["all", "valid"]:
        for inten in ["high", "medium", "low"]:
            rows.append({"signal": sig, "chan": chan, "intensity": inten,
                         **signed_rank(state_table(sig, (inten,), chan=chan), "")})
res_s4 = pd.DataFrame(rows).drop(columns="reading")
print(res_s4.round(6).to_string(index=False))

Result: reproduced, under the all-channels reading. EEG: rest larger at high (p = 0.0020) and
medium (p = 0.0039), not at low (p = 0.19, rest larger in 2 of 10). LFP: rest larger at all three
(p = 0.020, 0.0039 and 0.039). CSD: high (p = 0.020) and medium (p = 0.0078), not low (p = 0.16).
With the file's valid channels and trials the LFP low-intensity test gives p = 0.055, so under
that reading the LFP part would not hold.

## speed-threshold-robust

> The EEG, LFP and CSD state modulations remain when the speed threshold separating rest and
> active trials is raised up to 5 cm/s.

Thresholds 0.1, 0.5, 1, 2 and 5 cm/s (criterion's grid), reading (a)(i) of the base claims (high
intensity, all channels and trials), and the same with the file's valid channels and trials. A
session enters at a threshold when it has at least one trial in each state; at least 10 per state
is a variant. The file's own `is_running` (above 0 cm/s) is shown as well.

In [ ]:
rows = []
for sig in ["eeg", "lfp", "csd"]:
    for chan in ["all", "valid"]:
        for thr in russo.THRESHOLDS + ["file"]:
            t = state_table(sig, ("high",), thr, chan)
            t = t[(t.n_rest >= 1) & (t.n_active >= 1)]
            t10 = t[(t.n_rest >= 10) & (t.n_active >= 10)]
            a, b = signed_rank(t, ""), signed_rank(t10, "")
            rows.append({"signal": sig, "chan": chan, "threshold": thr, "n": a["n"], "rest_larger": a["rest_larger"],
                         "p": a["p"], "n_min10": b["n"], "p_min10": b["p"]})
res_thr = pd.DataFrame(rows)
print(res_thr.round(6).to_string(index=False))

# The criterion as written: at each threshold of the grid, EEG, LFP and CSD give p < 0.05 with rest larger
# (reading (a)(i), all channels and trials, sessions with at least one trial per state)
grid = res_thr[(res_thr.chan == "all") & res_thr.threshold.isin(russo.THRESHOLDS)].copy()
grid["passes"] = (grid.p < 0.05) & (grid.rest_larger > grid.n / 2)
print("\nCriterion, p < 0.05 with rest larger:")
print(grid.pivot(index="threshold", columns="signal", values="passes").to_string())
print(f"All three signals at every threshold: {bool(grid.passes.all())}")

Result: not reproduced. The EEG modulation holds at every threshold from 0.1 to 5 cm/s
(10 of 10 sessions, p = 0.00195 each). The LFP holds up to 2 cm/s (p = 0.039) and not at 5 cm/s
(p = 0.098). The CSD holds only at 0.1 cm/s (p = 0.0195); from 0.5 cm/s on p = 0.074 to 0.36,
with rest larger in 7 of 9 mice throughout. The criterion asks for EEG, LFP and CSD at every
threshold and has no clause for a partial result, so the failures of LFP and CSD in the check
above make the claim not reproduced, by the same rule as fig3e-lf-synchrony-reduced. With at
least 10 trials per state the EEG also loses significance at 5 cm/s (5 sessions, p = 0.0625).
This outcome was proposed after the second verification round and agreed by the third.

## figs5-early-component-mouse

> In mice the early (3 to 50 ms) EEG component is not modulated by state, while the early LFP
> amplitude is larger during running and the early CSD amplitude differs between states (the
> legend gives no direction for the CSD).

The three amplitude measures over 3 to 50 ms, with the readings of the late claims. The EEG part
is a claim of absence judged on the paper's test (two-sided signed-rank, p >= 0.05).

In [ ]:
for sig in ["eeg", "lfp", "csd"]:
    print(sig.upper(), "early 3-50 ms amplitude")
    print(readings_table(sig, win="early").round(6).to_string(index=False))
print("EEG early PLF (reported, paper p = 0.052)")
print(readings_table("eeg", meas="plf", win="early").round(6).to_string(index=False))

Result: reproduced, under reading (b). The early EEG amplitude does not differ by state under
any reading (p = 0.16 to 0.84). The early LFP amplitude is larger during running (rest larger in 2
of 9 mice, p = 0.0195; reading (b) 5 of 18, p = 0.0013). The early CSD differs at p = 0.048 with
high and medium (rest larger in 12 of 18, medians 1.85 and 1.87) but not at high alone (p =
0.055), so the CSD part holds under reading (b) only.

## figs6-posterior-no-modulation

Blocked (`data_missing`): no file of the dandiset has electrical stimulation of visual or
retrosplenial cortex or hippocampus (data check, target regions are MOs and SSp only).

## triphasic-pattern

> Deep MOs stimulation evokes a triphasic spiking pattern in MOs and SM-TH, an initial excitation
> within 10 ms, an off period of about 10 to 140 ms (127.8 +- 4.1 ms of silence) and a rebound
> excitation at 140 to 250 ms.

Population PSTH (1 ms bins) of responsive RS units in MOs and relay units in SM-TH at high
intensity, per mouse. Readings of the silence (criterion): (a) per mouse, the median over units of
the first-spike latency in 75 to 250 ms minus 10 ms, averaged over mice; (b) per mouse, the time
the population rate stays below its baseline mean minus 2 SD after the initial peak.

In [ ]:
def psth_shape(r, cls):
    c, rate, n = russo.population_psth(r, cls)
    if rate is None:
        return None
    base = rate[(c >= -0.5) & (c < -0.05)]
    mu, sd = base.mean(), base.std()
    init = rate[(c >= 0.002) & (c < 0.010)]
    off = rate[(c >= 0.010) & (c < 0.140)]
    reb = rate[(c >= 0.140) & (c < 0.250)]
    # reading (b): from the end of the initial peak (first bin after 2 ms back under the mean), time below mu - 2 SD
    k0 = np.searchsorted(c, 0.010)
    below = rate < mu - 2 * sd
    k = k0
    while k < len(rate) and not below[k] and c[k] < 0.05:
        k += 1
    start = k
    while k < len(rate) and (below[k] or np.all(below[k:k + 3])):
        k += 1
    silence_b = (c[k] - c[start]) * 1e3 if start < len(rate) and below[start] else 0.0
    return {"units": n, "baseline_Hz": mu, "initial_peak_Hz": init.max(), "initial_peak_ms": c[(c >= 0.002) & (c < 0.010)][init.argmax()] * 1e3,
            "frac_10_140_below_baseline": float((off < mu).mean()), "rebound_peak_Hz": reb.max(),
            "rebound_peak_ms": c[(c >= 0.140) & (c < 0.250)][reb.argmax()] * 1e3, "silence_b_ms": silence_b}


rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    for cls in ["RS", "relay"]:
        s = psth_shape(r, cls)
        if s:
            lat = H[(H.subject == r["subject"]) & (H.cls == cls)].groupby("unit").lat_reb.median()
            rows.append({"subject": r["subject"], "cls": cls, **s, "silence_a_ms": (lat.median() - 0.010) * 1e3})
tri = pd.DataFrame(rows)
print(tri.round(2).to_string(index=False))
print("\nMean over mice:")
print(tri.groupby("cls")[["initial_peak_Hz", "baseline_Hz", "frac_10_140_below_baseline", "rebound_peak_Hz",
                          "rebound_peak_ms", "silence_a_ms", "silence_b_ms"]].mean().round(2).to_string())
print("Paper: silence 127.8 +- 4.1 ms")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for k, cls in enumerate(["RS", "relay"]):
    for r in S.values():
        if r.get("has_probes"):
            c, rate, n = russo.population_psth(r, cls)
            if rate is not None:
                sm = np.convolve(rate, np.ones(5) / 5, "same")
                ax[k].plot(c * 1e3, sm, lw=0.8)
    ax[k].set_xlim(-100, 400); ax[k].set_ylim(0, 150); ax[k].set_title(f"{cls} units, high intensity, per mouse"); ax[k].set_xlabel("ms")
ax[0].set_ylabel("rate (Hz), 5 ms running mean")
plt.tight_layout(); plt.show()
print("Smoothed rates above 150 Hz (outside the axes), by class:", {cls: int(sum((np.convolve(russo.population_psth(r, cls)[1], np.ones(5) / 5, "same") > 150).sum()
      for r in S.values() if r.get("has_probes") and russo.population_psth(r, cls)[1] is not None)) for cls in ["RS", "relay"]}, "bins")

In [ ]:
def silence_b(c, rate, smooth_ms):
    """Reading (b) on the PSTH smoothed by a running mean over smooth_ms 1 ms bins: from the first bin
    after 10 ms below the baseline mean minus 2 SD (baseline -500 to -50 ms of the smoothed PSTH),
    the time until it comes back above."""
    sm = np.convolve(rate, np.ones(smooth_ms) / smooth_ms, "same") if smooth_ms > 1 else rate
    base = sm[(c >= -0.5) & (c < -0.05)]
    mu, sd = base.mean(), base.std()
    below = sm < mu - 2 * sd
    k = np.searchsorted(c, 0.010)
    while k < len(sm) and not below[k] and c[k] < 0.05:
        k += 1
    if k >= len(sm) or not below[k]:
        return mu, sd, 0.0
    start = k
    while k < len(sm) and below[k]:
        k += 1
    return mu, sd, (c[k] - c[start]) * 1e3


SMOOTH = [1, 2, 5, 10, 20]
rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    for cls in ["RS", "relay"]:
        c, rate, n = russo.population_psth(r, cls)
        if rate is None:
            continue
        row = {"subject": r["subject"], "cls": cls, "units": n}
        for k in SMOOTH:
            mu, sd, s = silence_b(c, rate, k)
            if k in (1, 10):
                row[f"base_mean_{k}ms"], row[f"base_sd_{k}ms"], row[f"threshold_{k}ms"] = mu, sd, mu - 2 * sd
            row[f"silence_{k}ms"] = s
        rows.append(row)
sb = pd.DataFrame(rows)
print("Baseline mean, SD and threshold (mean - 2 SD) of the population rate, Hz, at 1 ms and smoothed over 10 ms:")
print(sb[["subject", "cls", "units", "base_mean_1ms", "base_sd_1ms", "threshold_1ms", "base_mean_10ms", "base_sd_10ms",
          "threshold_10ms"]].round(2).to_string(index=False))
print("\nReading (b), silence (ms) by smoothing width:")
print(sb[["subject", "cls", "units"] + [f"silence_{k}ms" for k in SMOOTH]].round(1).to_string(index=False))
print("\nMean over mice (ms); paper 127.8 +- 4.1, so 123.7 to 131.9:")
print(sb.groupby("cls")[[f"silence_{k}ms" for k in SMOOTH]].mean().round(1).to_string())
print("Without the relay population of 551399 (one unit):",
      sb[(sb.cls == "relay") & (sb.subject != "551399")][[f"silence_{k}ms" for k in SMOOTH]].mean().round(1).to_dict())

Result: partially reproduced. The triphasic pattern is present in every mouse: the initial
peak falls at 2.5 to 9.5 ms, the rate is below baseline over a mean of 99% (RS) and 88% (relay)
of 10 to 140 ms, and the rebound peaks at 150 to 194 ms. The silence does not match 127.8 +- 4.1
ms under either reading of the criterion: (a) 171.5 ms (RS) and 160.8 ms (relay), (b) 57.9 and
71.4 ms. Reading (b) on 1 ms bins is degenerate: with baseline rates of 2.6 to 8.8 Hz the
threshold, baseline mean minus 2 SD, is 0.05 to 6.51 Hz and below zero for the one-unit relay
population of 551399, and 7 of the 16 populations give silences of 0 to 5 ms. On PSTHs smoothed by a running mean,
the RS silence is 113.8 ms (2 ms), 134.1 (5 ms), 132.8 (10 ms) and 127.9 ms (20 ms), within
123.7 to 131.9 ms only at 20 ms; the relay silence stays at 86.4 to 98.7 ms (100.8 to 107.3 ms
without 551399). So the RS value comes near the paper's at wider smoothing, and the relay value
does not at any width.

## fig2b-thalamus-leads

> The late rebound of SM-TH units precedes that of MOs RS units and the late CSD, LFP and EEG
> peaks, at rest and during running.

Per mouse at high intensity (criterion): SM-TH and MOs latency, per trial the median over
responsive relay (SM-TH) or RS (MOs) units of the first-spike latency in 75 to 250 ms, averaged
over trials; EEG, LFP and CSD latency, per trial the largest GFP peak in 75 to 250 ms whose
prominence exceeds the baseline mean plus 3 SD, averaged over trials. Two-way ANOVA on measure and
state, Tukey HSD on the measure. Tukey within each state is shown as well.

In [ ]:
def trial_unit_latency(cls, col="lat_reb", frame=H):
    """Per mouse and trial, the median over the class's responsive units of the latency."""
    return frame[frame.cls == cls].groupby(["subject", "trial"])[col].median()


rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    hi = (t.intensity == "high") & t.is_valid
    per = {"EEG": pd.Series(r["eeg_lat"]), "LFP": pd.Series(r["lfp_lat"]), "CSD": pd.Series(r["csd_lat"])}
    for name, cls in [("SM-TH", "relay"), ("MOs", "RS")]:
        s = trial_unit_latency(cls)
        if r["subject"] in s.index.get_level_values(0):
            per[name] = s.loc[r["subject"]].reindex(range(len(t)))
    for name, lat in per.items():
        for st, m in [("rest", hi & ~t.active), ("active", hi & t.active)]:
            v = lat[m.to_numpy()].dropna()
            if len(v):
                rows.append({"subject": r["subject"], "measure": name, "state": st, "latency": v.mean(), "trials": len(v)})
lat2b = pd.DataFrame(rows)
print("Mean latency (ms) over mice, and number of mice:")
print((lat2b.groupby(["measure", "state"]).latency.agg(["mean", "std", "size"]) * [1e3, 1e3, 1]).round(1).to_string())
fit = smf.ols("latency ~ C(measure) * C(state)", data=lat2b).fit()
print("\nTwo-way ANOVA (type II):")
print(anova_lm(fit, typ=2).to_string())
tk = pairwise_tukeyhsd(lat2b.latency * 1e3, lat2b.measure)
print("\nTukey HSD on measure (ms):")
print(tk.summary())
for st in ["rest", "active"]:
    d = lat2b[lat2b.state == st]
    print(f"\nTukey HSD on measure, {st} only:")
    print(pairwise_tukeyhsd(d.latency * 1e3, d.measure).summary())

In [ ]:
# Variant: the EEG and LFP GFP over the channels the file marks valid (the CSD has no such flag)
rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    hi = (t.intensity == "high") & t.is_valid
    per = {"EEG": pd.Series(r["eeg_lat_valid"]), "LFP": pd.Series(r["lfp_lat_valid"]), "CSD": pd.Series(r["csd_lat"])}
    for name, cls in [("SM-TH", "relay"), ("MOs", "RS")]:
        s = trial_unit_latency(cls)
        if r["subject"] in s.index.get_level_values(0):
            per[name] = s.loc[r["subject"]].reindex(range(len(t)))
    for name, lat in per.items():
        for st, m in [("rest", hi & ~t.active), ("active", hi & t.active)]:
            v = lat[m.to_numpy()].dropna()
            if len(v):
                rows.append({"subject": r["subject"], "measure": name, "state": st, "latency": v.mean(), "trials": len(v)})
lat2b_valid = pd.DataFrame(rows)
print("Valid channels only. Mean latency (ms) over mice, and number of mice:")
print((lat2b_valid.groupby(["measure", "state"]).latency.agg(["mean", "std", "size"]) * [1e3, 1e3, 1]).round(1).to_string())
print("\nTwo-way ANOVA (type II):")
print(anova_lm(smf.ols("latency ~ C(measure) * C(state)", data=lat2b_valid).fit(), typ=2).to_string())
print("\nTukey HSD on measure (ms):")
print(pairwise_tukeyhsd(lat2b_valid.latency * 1e3, lat2b_valid.measure).summary())

Result: not reproduced. The measure factor of the two-way ANOVA is not significant (F = 1.41,
p = 0.24) and no Tukey comparison of SM-TH with another measure is (adjusted p 0.29 to 0.84).
SM-TH has the shortest mean latency during running (146.4 ms against 158.0 to 163.3 for the
others) but not at rest (173.4 against 172.3 to 176.1). With the EEG and LFP GFP over the valid
channels only, the measure factor gives p = 0.38 and the Tukey comparisons of SM-TH adjusted p
0.57 to 0.83. Only 7 mice have responsive relay units, against 20 in the paper.

## figs7a-active-shorter-latency

> Late responses have shorter latencies during running than at rest for all measures.

In the data of fig2b-thalamus-leads: reproduced when the mean latency is shorter in active than
rest for all five measures; the state factor of the ANOVA is reported.

In [ ]:
m7 = lat2b.groupby(["measure", "state"]).latency.mean().unstack() * 1e3
m7["active_minus_rest_ms"] = m7.active - m7.rest
paired = lat2b.pivot_table(index=["subject", "measure"], columns="state", values="latency").dropna()
m7["mice_active_shorter"] = (paired.active < paired.rest).groupby("measure").sum()
m7["mice"] = paired.groupby("measure").size()
print(m7.round(2).to_string())
print("\nState factor of the two-way ANOVA: F = %.2f, p = %.4g" % tuple(anova_lm(fit, typ=2).loc["C(state)", ["F", "PR(>F)"]]))

Result: reproduced. The mean latency is shorter during running for all five measures (by 12.3
to 27.0 ms), in 7 to 9 mice per measure, and the state factor of the ANOVA gives F = 39.4, p =
2.1e-08.

## figs7b-latency-correlation

> On single trials, the later the SM-TH units rebound, the later the MOs units and the CSD, LFP
> and EEG peaks.

Per trial at high intensity, the latencies of fig2b-thalamus-leads before averaging; four mixed
models Measure ~ ThalamicUnits + (1 | MouseID) (statsmodels mixedlm).

In [ ]:
rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    th = trial_unit_latency("relay")
    if r["subject"] not in th.index.get_level_values(0):
        continue
    d = pd.DataFrame({"TH": th.loc[r["subject"]].reindex(range(len(t))),
                      "MOs": trial_unit_latency("RS").loc[r["subject"]].reindex(range(len(t))),
                      "EEG": r["eeg_lat"], "LFP": r["lfp_lat"], "CSD": r["csd_lat"]})
    d["subject"] = r["subject"]
    d = d[((t.intensity == "high") & t.is_valid).to_numpy()]
    rows.append(d)
pt7 = pd.concat(rows) * 1
for c in ["TH", "MOs", "EEG", "LFP", "CSD"]:
    pt7[c] = pt7[c] * 1e3
out = []
for m in ["MOs", "CSD", "LFP", "EEG"]:
    d = pt7.dropna(subset=["TH", m])
    fm = smf.mixedlm(f"{m} ~ TH", d, groups=d["subject"]).fit()
    rho = stats.spearmanr(d.TH, d[m])
    out.append({"measure": m, "trials": len(d), "mice": d.subject.nunique(), "slope": fm.params["TH"],
                "p": fm.pvalues["TH"], "spearman_r": rho.statistic, "spearman_p": rho.pvalue})
print(pd.DataFrame(out).to_string(index=False))

Result: reproduced. On single trials the SM-TH rebound latency predicts the MOs, CSD, LFP and
EEG latencies with positive slopes 0.30 to 0.43 (mixed models, p = 4e-20 to 9e-71; 547 to 796
trials from 7 mice).

## fig3a-thalamic-rate-higher-active

> The evoked firing rate of SM-TH units in the late window (150 to 250 ms) is higher during
> running than at rest, unlike the late EP component.

Responsive relay units at high intensity; per unit and trial, spikes in 150 to 250 ms over 0.1 s.
Readings (criterion): (a) per unit, signed-rank on unit pairs; (b) per mouse, the median over its
units, signed-rank; (c) per trial, the mean over the session's units, rank-sum on rest against
active trials pooled over sessions. Variant: the code's rank-sum on per-session medians. As the
sessions differ in rate and in their fractions of active trials, reading (c) is also shown within
each session and as a session-stratified rank-sum (van Elteren); these are checks, not readings of
the criterion.

In [ ]:
def rate_readings(cls, frame=H, col="rate_late"):
    d = frame[frame.cls == cls]
    pu = d.groupby(["subject", "unit", "active"])[col].mean().unstack().dropna()
    pm = pu.groupby("subject").median()
    pt = d.groupby(["subject", "trial", "active"])[col].mean().reset_index()
    out = []
    w = stats.wilcoxon(pu[False], pu[True])
    out.append({"reading": "(a) per unit, signed-rank", "n": len(pu), "rest": pu[False].mean(), "active": pu[True].mean(), "p": w.pvalue})
    if len(pm) > 1:
        w = stats.wilcoxon(pm[False], pm[True])
        out.append({"reading": "(b) per mouse median, signed-rank", "n": len(pm), "rest": pm[False].median(), "active": pm[True].median(), "p": w.pvalue})
    a, rr = pt[pt.active][col], pt[~pt.active][col]
    out.append({"reading": "(c) per trial, rank-sum", "n": f"{len(rr)} rest, {len(a)} active", "rest": rr.mean(), "active": a.mean(),
                "p": stats.mannwhitneyu(rr, a).pvalue})
    sm = d.groupby(["subject", "active"])[col].median().unstack()
    out.append({"reading": "code variant, per-session medians, rank-sum", "n": len(sm), "rest": sm[False].median(), "active": sm[True].median(),
                "p": stats.mannwhitneyu(sm[False].dropna(), sm[True].dropna()).pvalue})
    return pd.DataFrame(out), pu


r3a, pu3a = rate_readings("relay")
print(r3a.to_string(index=False))
print("\nPer mouse (median over units, Hz):")
print(pu3a.groupby("subject").agg(["median", "size"]).round(2).to_string())

In [ ]:
def van_elteren(d, col):
    """Session-stratified rank-sum test (van Elteren 1960): rest against active trials within each
    session, rank sums weighted by 1/(n+1), normal approximation with the tie correction."""
    num, var = 0.0, 0.0
    for _, s in d.groupby("subject"):
        a, r = s[s.active][col].to_numpy(), s[~s.active][col].to_numpy()
        if len(a) == 0 or len(r) == 0:
            continue
        n = len(a) + len(r)
        ranks = stats.rankdata(np.r_[a, r])
        _, cnt = np.unique(np.r_[a, r], return_counts=True)
        v = len(a) * len(r) / 12 * ((n + 1) - (cnt ** 3 - cnt).sum() / (n * (n - 1)))
        w = 1 / (n + 1)
        num += w * (ranks[:len(a)].sum() - len(a) * (n + 1) / 2)
        var += w ** 2 * v
    z = num / np.sqrt(var)
    return z, 2 * stats.norm.sf(abs(z))


def per_session_rates(cls, frame=H, col="rate_late"):
    """Reading (c) within each session: per trial the mean over the session's units, rest against active."""
    d = frame[frame.cls == cls]
    pt = d.groupby(["subject", "trial", "active"])[col].mean().reset_index()
    rows = []
    for s, g in pt.groupby("subject"):
        a, r = g[g.active][col], g[~g.active][col]
        rows.append({"subject": s, "units": d[d.subject == s].unit.nunique(), "trials_rest": len(r), "trials_active": len(a),
                     "rest_Hz": r.mean(), "active_Hz": a.mean(),
                     "p_ranksum": stats.mannwhitneyu(r, a).pvalue if len(a) and len(r) else np.nan})
    z, p = van_elteren(pt, col)
    return pd.DataFrame(rows), z, p


t3a, z, p = per_session_rates("relay")
print("Relay units, per session (reading (c) within each session):")
print(t3a.round(4).to_string(index=False))
print(f"van Elteren, stratified by session: z = {z:.2f} (positive: active larger), p = {p:.2g}")
print(f"Sessions with active larger: {int((t3a.active_Hz > t3a.rest_Hz).sum())} of {len(t3a)}; "
      f"significant (p < 0.05) active larger {int(((t3a.active_Hz > t3a.rest_Hz) & (t3a.p_ranksum < 0.05)).sum())}, "
      f"rest larger {int(((t3a.active_Hz < t3a.rest_Hz) & (t3a.p_ranksum < 0.05)).sum())}")

Result: reproduced, under reading (c) only. Per trial, the mean late rate of responsive relay
units is higher during running (17.2 against 13.9 Hz; rank-sum p = 0.00054, 681 rest and 150
active trials). Per unit (303 units, 14.8 against 14.4 Hz, p = 0.41) and per mouse (7 mice, p =
0.30) the difference is not significant, nor with the code's per-session medians (p = 0.24). The
pooled per-trial test mixes sessions with different rates and fractions of active trials. Within
sessions the rate is higher during running in 5 of 7 (significantly in 4: 551399, 569068, 569073,
571619) and significantly lower in 551397 (17.3 against 13.5 Hz); the session-stratified van
Elteren test gives p = 0.19. The criterion accepts reading (c), so the outcome stands, but it does
not hold within sessions taken together.

## figs9-mos-rate-no-modulation

> The late evoked firing rate of MOs units is not modulated by state.

Claim of absence, on the paper's test: responsive RS units in MOs, one median per mouse and
state, two-sided signed-rank across mice. All MOs units (RS and FS) are a variant, since the
Figure S9A legend says MOs units.

In [ ]:
r9, pu9 = rate_readings("RS")
print("RS units")
print(r9.to_string(index=False))
Hm = H.assign(cls=H.cls.replace({"FS": "MOs_all", "RS": "MOs_all"}))
r9b, _ = rate_readings("MOs_all", frame=Hm)
print("\nVariant, all responsive MOs units")
print(r9b.to_string(index=False))

Result: not reproduced. On the paper's test, per-mouse medians of responsive RS units, the late
rate differs by state (9 mice, 3.65 against 3.33 Hz, p = 0.0117), lower during running. All
responsive MOs units give p = 0.027, and per unit p = 6e-10.

## figs10a-fs-trn-no-modulation

> The late evoked firing of putative cortical interneurons (FS) and TRN units is not modulated by
> state.

Claim of absence. Responsive FS units in MOs and thalamic units under 0.35 ms, with the reading of
the unit of test adopted in fig3a-thalamic-rate-higher-active, (c), and all readings shown; the
same within-session and session-stratified checks as there. TRN is the SM-TH units under 0.35 ms;
the variant TRN+RT adds the units under 0.35 ms located in RT itself.

In [ ]:
for cls in ["FS", "TRN"]:
    r10, _ = rate_readings(cls)
    print(cls)
    print(r10.to_string(index=False))

In [ ]:
for cls in ["FS", "TRN"]:
    t, z, p = per_session_rates(cls)
    print(f"{cls} units, per session (reading (c) within each session):")
    print(t.round(4).to_string(index=False))
    print(f"van Elteren, stratified by session: z = {z:.2f} (positive: active larger), p = {p:.2g}\n")
# Variant: TRN also counts the units under 0.35 ms located in RT itself (TRN-RT)
Hrt = H[H.cls.isin(["TRN", "TRN-RT"])].assign(cls="TRN+RT")
r10rt, _ = rate_readings("TRN+RT", frame=Hrt)
print("TRN with RT units:")
print(r10rt.to_string(index=False))
t, z, p = per_session_rates("TRN+RT", frame=Hrt)
print(t.round(4).to_string(index=False))
print(f"van Elteren, stratified by session: z = {z:.2f} (positive: active larger), p = {p:.2g}")

# The criterion as written (reading (c), the one adopted in fig3a), next to the within-session check, for fig3a too
rows = []
for claim, cls, frame in [("fig3a (relay, active larger)", "relay", H), ("figs10a FS", "FS", H), ("figs10a TRN", "TRN", H)]:
    rr = rate_readings(cls, frame=frame)[0].set_index("reading").loc["(c) per trial, rank-sum"]
    _, z, p = per_session_rates(cls, frame=frame)
    rows.append({"claim": claim, "rest_Hz": rr.rest, "active_Hz": rr.active, "p_reading_c": rr.p, "van_Elteren_z": z, "p_van_Elteren": p})
crit = pd.DataFrame(rows)
print("\nReading (c), pooled over sessions, against the session-stratified test:")
print(crit.round(4).to_string(index=False))
print(f"figs10a by the criterion (neither FS nor TRN p < 0.05 under reading (c)): "
      f"{'reproduced' if (crit.p_reading_c[1:] >= 0.05).all() else 'not reproduced'}")

Result: reproduced, by the criterion, which fixes the test as the per-trial reading (c) adopted
for fig3a-thalamic-rate-higher-active: neither FS (22.1 against 20.3 Hz, p = 0.35) nor TRN (31.2
against 30.3 Hz, p = 0.88) differs by state (table above). This outcome was proposed after the
second verification round and agreed by the third. It is fragile on two counts.

First, FS rates are lower during running per unit and per mouse (p = 0.0068 and 0.039), and
within sessions they are lower in 6 of 8 (significantly in 5), with van Elteren p = 1.3e-08; in
569069 they are significantly higher during running (27.6 against 22.8 Hz, p = 0.0033). The pooled
test hides this because the sessions differ in rate and in their fractions of active trials, as
for fig3a (van Elteren p = 0.19).

Second, the outcome depends on TRN being the SM-TH units under 0.35 ms. Adding the one responsive
RT unit (546655) to TRN makes the pooled per-trial TRN test significant (30.3 against 22.0 Hz, p =
8e-06), which would make the claim not reproduced. This comes from the same mixing (that session
has 81 of 119 trials active); stratified by session it gives p = 0.77, and per unit and per mouse
p = 0.47 and 0.69. TRN on the SM-TH class does not differ under any reading (p = 0.31 to 1) nor
within sessions (van Elteren p = 0.39).

## fig3b-bimodal-isi

> The evoked late ISI of putative thalamic relay units is bimodal, splitting them into
> high-firing (HF, ISI below 17.88 ms, 569 units) and low-firing (LF, 700 units) groups, while
> the ISI distributions of cortical RS, cortical FS and TRN units are unimodal.

Evoked ISI per unit: the median over trials of the interval between the first spike and the next
in the window, (a) 150 to 250 ms (Methods) or (b) 75 to 250 ms (code). k-means with two clusters
on the log ISI of the relay units, seeded from the claim; threshold the largest ISI of the short
cluster. Hartigan's dip test on log10 ISI per class, with the `diptest` package (tabulated
p-values); the paper's code used `unidip`, which does not run with current numpy.

In [ ]:
SEED_3B = claim_seed("fig3b-bimodal-isi")


def isi_split(col, seed):
    isi = H.groupby(["subject", "unit", "cls"])[col].median().dropna().reset_index()
    rel = isi[isi.cls == "relay"].copy()
    km = KMeans(2, n_init=10, random_state=seed).fit(np.log(rel[col].to_numpy() * 1e3).reshape(-1, 1))
    short = int(np.argmin(km.cluster_centers_.ravel()))
    rel["group"] = np.where(km.labels_ == short, "HF", "LF")
    thr = rel.loc[rel.group == "HF", col].max() * 1e3
    return isi, rel, thr


res3b = {}
for label, col in [("(a) 150-250 ms", "isi_late"), ("(b) 75-250 ms", "isi_reb")]:
    isi, rel, thr = isi_split(col, SEED_3B)
    res3b[label] = (isi, rel, thr)
    dips = {c: diptest.diptest(np.log10(isi[isi.cls == c][col].to_numpy() * 1e3)) for c in ["relay", "TRN", "RS", "FS"]}
    print(f"{label}: units per class {isi.groupby('cls').size().to_dict()}")
    print(f"  k-means: HF {int((rel.group == 'HF').sum())}, LF {int((rel.group == 'LF').sum())}, threshold {thr:.2f} ms")
    print("  dip test (dip, p):", {c: (round(d, 4), round(p, 4)) for c, (d, p) in dips.items()})
    print("  relay ISI percentiles 5, 25, 50, 75, 95 (ms):", np.round(np.percentile(rel[col] * 1e3, [5, 25, 50, 75, 95]), 2))
    print(f"  relay units with ISI above 17.88 ms: {int((rel[col] * 1e3 > 17.88).sum())} of {len(rel)}")
    seeds = []
    for k in range(5):
        _, rk, tk_ = isi_split(col, SEED_3B + 1000 * k)
        seeds.append((int((rk.group == "HF").sum()), round(tk_, 2)))
    print("  HF count and threshold under seeds +0, +1000, ..., +4000:", seeds)

fig, ax = plt.subplots(1, 4, figsize=(13, 2.6))
isi, rel, thr = res3b["(a) 150-250 ms"]
for k, c in enumerate(["relay", "TRN", "RS", "FS"]):
    v = np.log10(isi[isi.cls == c].isi_late * 1e3)
    ax[k].hist(v, 30); ax[k].set_title(f"{c} (n = {len(v)})"); ax[k].set_xlabel("log10 evoked ISI (ms)")
    ax[k].axvline(np.log10(17.88), color="r", lw=0.8)
plt.tight_layout(); plt.show()
print("Plotted log10 ISI range:", round(np.log10(isi.isi_late * 1e3).min(), 2), "to", round(np.log10(isi.isi_late * 1e3).max(), 2))

In [ ]:
# Variant: TRN also counts the units under 0.35 ms located in RT itself (TRN-RT)
for label, (isi, rel, thr) in res3b.items():
    col = "isi_late" if label.startswith("(a)") else "isi_reb"
    v = isi[isi.cls.isin(["TRN", "TRN-RT"])][col].to_numpy() * 1e3
    d, p = diptest.diptest(np.log10(v))
    print(f"{label}: TRN with RT units, {len(v)} units ({int((isi.cls == 'TRN-RT').sum())} from RT), dip {d:.4f}, p = {p:.4f}")

Result: not reproduced. The evoked ISI of the 296 relay units is not bimodal (dip test p =
0.73 for 150 to 250 ms, 0.64 for 75 to 250 ms); the median is 2.6 ms and only 8 units have an ISI
above 17.88 ms. k-means splits off a small long-ISI group (HF 278, LF 18, threshold 6.9 ms; 281,
15 and 7.5 ms in the wider window). TRN, RS and FS are unimodal (p = 0.15 to 0.95), and TRN with
the RT unit too (p = 0.80 and 0.18). The counts and threshold under the four other seeds are 277
to 281 and 6.6 to 7.5 ms. Every HF and LF claim below is tested on these groups: an LF group of
18 units (8 with the 17.88 ms threshold) from 4 mice, against 700 LF units in the paper.

## figs11d-hf-earlier-less-variable

> HF units respond earlier and with less latency variability than LF units in the early window
> (3 to 50 ms).

HF and LF as split in fig3b-bimodal-isi (reading (a) window; the paper's 17.88 ms threshold as a
variant). Per unit, the median over trials of the first-spike latency in 3 to 50 ms and its SD.
Readings (criterion): (a) per mouse medians, signed-rank on the mouse pairs; (b) rank-sum on unit
values.

In [ ]:
def hf_lf_groups(split):
    if split == "k-means (a)":
        rel = res3b["(a) 150-250 ms"][1]
        return rel.set_index(["subject", "unit"]).group
    rel = res3b["(a) 150-250 ms"][1].copy()
    rel["group"] = np.where(rel.isi_late * 1e3 < 17.88, "HF", "LF")
    return rel.set_index(["subject", "unit"]).group


rows = []
for split in ["k-means (a)", "17.88 ms"]:
    g = hf_lf_groups(split)
    d = H[H.cls == "relay"].groupby(["subject", "unit"]).lat_early.agg(["median", "std"]).join(g, how="inner")
    for meas in ["median", "std"]:
        hf, lf = d[d.group == "HF"][meas].dropna(), d[d.group == "LF"][meas].dropna()
        pm = d.groupby(["subject", "group"])[meas].median().unstack().dropna()
        rows.append({"split": split, "measure": "latency" if meas == "median" else "latency SD",
                     "HF_units": len(hf), "LF_units": len(lf), "HF_median_ms": hf.median() * 1e3, "LF_median_ms": lf.median() * 1e3,
                     "p_ranksum_units": stats.mannwhitneyu(hf, lf).pvalue if len(lf) and len(hf) else np.nan,
                     "mice_with_both": len(pm), "p_signedrank_mice": stats.wilcoxon(pm.HF, pm.LF).pvalue if len(pm) > 1 else np.nan,
                     "mice_HF_lower": int((pm.HF < pm.LF).sum())})
print(pd.DataFrame(rows).round(5).to_string(index=False))

Result: not reproduced. With the k-means split HF units respond later than LF units in 3 to
50 ms (median 8.24 against 6.69 ms, rank-sum p = 0.00017), the reverse of the claim; their latency
SD is lower (2.59 against 3.48 ms) but not significantly (p = 0.12). Per mouse only 4 mice have
both groups (p = 0.88 and 0.125). The 17.88 ms split gives the same direction.

## figs12-intensity-shifts-to-hf

> Higher stimulation intensities recruit more responsive thalamic units and a larger share of HF
> relative to LF units, LF units shifting to HF as intensity increases.

Units classed HF or LF by the 17.88 ms threshold at each intensity, the ISI per unit computed at
that intensity with the window of fig3b reading (a) (criterion).

In [ ]:
R = U[U.responsive & (U.cls == "relay")]
isi_i = R.groupby(["intensity", "subject", "unit"]).isi_late.median().dropna().reset_index()
isi_i["HF"] = isi_i.isi_late * 1e3 < 17.88
tab = R.groupby("intensity").apply(lambda d: d.groupby(["subject", "unit"]).ngroups).rename("responsive_units").to_frame()
tab["units_with_isi"] = isi_i.groupby("intensity").size()
tab["HF_fraction"] = isi_i.groupby("intensity").HF.mean()
print(tab.loc[["low", "medium", "high"]].round(3).to_string())
pm12 = isi_i.groupby(["subject", "intensity"]).HF.mean().unstack()
print("\nHF fraction per mouse:")
print(pm12[["low", "medium", "high"]].round(3).to_string())
both = pm12[["low", "high"]].dropna()
print(f"Mice with units at low and high: {len(both)}; HF fraction higher at high than low in {int((both.high > both.low).sum())}, "
      f"equal in {int((both.high == both.low).sum())}")
print("\nRelay units with an evoked ISI behind each mouse's HF fraction:")
print(isi_i.groupby(["subject", "intensity"]).size().unstack(fill_value=0)[["low", "medium", "high"]].to_string())

Result: reproduced. Responsive relay units increase from 13 (low) to 216 (medium) to 303
(high), and the HF fraction from 0.600 to 0.972 to 0.973; per mouse the HF fraction is higher at
high than at low intensity in 3 of the 4 mice with units at both. The increase from medium to
high is 0.001, and the low-intensity value rests on 10 units: 6, 2, 1 and 1 in the four mice.

## fig3e-baseline-rate-doubles

> During running the baseline firing rate of thalamic relay units almost doubles (6.78 +- 4.4 Hz
> at rest, 12.48 +- 4.6 Hz active), for both HF and LF units.

Baseline rate per unit and trial: spikes in -1 to -0.01 s over 0.99 s, responsive relay units at
high intensity, HF and LF as split in fig3b-bimodal-isi (k-means, reading (a)); the 17.88 ms
threshold is a variant. Readings (criterion): (a) per trial, the mean over the group's units,
rest against active trials pooled over sessions, rank-sum, Bonferroni over HF and LF; the mean and
SD are over the HF and LF trial values pooled; (b) per unit, mean over rest and active trials,
signed-rank.

In [ ]:
def grouped(split):
    g = hf_lf_groups(split).rename("group")
    return H[H.cls == "relay"].join(g, on=["subject", "unit"], how="inner")


def per_trial(df, func):
    """Per session, trial and group: a value over the group's units in that trial."""
    return df.groupby(["subject", "trial", "group", "active"]).apply(func).rename("v").dropna().reset_index()


def feature_tests(pt, pu=None, bonf=2):
    rows = []
    for grp in ["HF", "LF"]:
        d = pt[pt.group == grp]
        rest, act = d[~d.active].v, d[d.active].v
        p = stats.mannwhitneyu(rest, act).pvalue if len(rest) and len(act) else np.nan
        row = {"group": grp, "trials_rest": len(rest), "trials_active": len(act), "rest_mean": rest.mean(), "rest_sd": rest.std(),
               "active_mean": act.mean(), "active_sd": act.std(), "p_ranksum_trials_bonf": min(1, p * bonf)}
        if pu is not None:
            u = pu[pu.group == grp]
            w = stats.wilcoxon(u[False], u[True]) if len(u) > 1 else None
            row.update({"units_or_mice": len(u), "rest_med_b": u[False].median(), "active_med_b": u[True].median(),
                        "p_signedrank_b_bonf": min(1, w.pvalue * bonf) if w else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)


for split in ["k-means (a)", "17.88 ms"]:
    G = grouped(split)
    pt = per_trial(G, lambda d: d.rate_base.mean())
    pu = G.groupby(["group", "subject", "unit", "active"]).rate_base.mean().unstack().dropna().reset_index()
    print(f"HF/LF split: {split}")
    print(feature_tests(pt, pu).round(5).to_string(index=False))
    both = pt.groupby("active").v.agg(["mean", "std", "size"])
    print("HF and LF trial values pooled (Hz): rest %.2f +- %.2f (n %d), active %.2f +- %.2f (n %d); paper 6.78 +- 4.4 and 12.48 +- 4.6"
          % (both.loc[False, "mean"], both.loc[False, "std"], both.loc[False, "size"], both.loc[True, "mean"], both.loc[True, "std"], both.loc[True, "size"]))
    allu = per_trial(H[H.cls == "relay"].assign(group="all"), lambda d: d.rate_base.mean())
    print("All responsive relay units, per trial (Hz): rest %.2f, active %.2f\n" % (allu[~allu.active].v.mean(), allu[allu.active].v.mean()))

Result: reproduced. Per trial, the baseline rate of relay units rises from 6.82 +- 5.38 Hz at
rest to 14.60 +- 7.39 Hz during running (HF and LF trial values pooled; paper 6.78 +- 4.4 and
12.48 +- 4.6), both within tolerance, and both HF (5.20 to 12.45 Hz) and LF (9.83 to 17.75 Hz)
rise with corrected p < 1e-5. Per unit the same holds (p < 0.0001). With the 17.88 ms split the
active mean is 15.27 Hz, also within tolerance.

## fig3e-lf-synchrony-reduced

> During running the rebound onset variability (ROV) of LF units increases (35.7 +- 11.8 to
> 40.7 +- 8.2 ms), a loss of synchrony, while the ROV of HF units does not change.

ROV per trial: the SD across the group's units of the first-spike latency, in (a) 150 to 250 ms
(Methods) or (b) 75 to 250 ms (code); trials with at least two units firing. Unit of test
(criterion): per trial, rank-sum (Bonferroni over HF and LF); per mouse, the mean ROV of its rest
and active trials, signed-rank.

In [ ]:
def rov(col):
    return lambda d: d[col].std() * 1e3 if d[col].notna().sum() >= 2 else np.nan


for split in ["k-means (a)", "17.88 ms"]:
    G = grouped(split)
    for label, col in [("(a) 150-250 ms", "lat_late"), ("(b) 75-250 ms", "lat_reb")]:
        pt = per_trial(G, rov(col))
        pm = pt.groupby(["group", "subject", "active"]).v.mean().unstack().dropna().reset_index()
        print(f"split {split}, window {label} (ms)")
        print(feature_tests(pt, pm).round(5).to_string(index=False))
        print()

Result: not reproduced. No reading meets the criterion. In 75 to 250 ms with the k-means split
the LF onset variability rises during running (44.3 to 57.8 ms, per-trial rank-sum corrected p =
4e-05) and the HF one does not (31.7 and 31.9, p = 1), but the active LF mean, 57.8 ms, is outside
40.7 +- 8.2 ms. In 150 to 250 ms LF does not change (26.3 and 25.8 ms, p = 1), nor with the paper's
17.88 ms split in either window. Per mouse (4 mice with LF units) no change is significant. The LF
group has 18 units.

## fig3e-hf-frequency-reduced

> During running the late response frequency of HF units falls (309 +- 126 to 241 +- 116 Hz),
> while that of LF units does not change significantly.

Response frequency: per trial, the median ISI across the group's units, inverted (code, criterion);
windows as in fig3e-lf-synchrony-reduced. Unit of test: per trial, rank-sum; per unit, the mean
frequency over rest and over active trials, signed-rank.

In [ ]:
for split in ["k-means (a)", "17.88 ms"]:
    G = grouped(split)
    for label, col in [("(a) 150-250 ms", "isi_late"), ("(b) 75-250 ms", "isi_reb")]:
        pt = per_trial(G, lambda d: 1.0 / d[col].median() if d[col].notna().any() else np.nan)
        pu = (1.0 / G[col]).groupby([G.group, G.subject, G.unit, G.active]).mean().unstack().dropna().reset_index()
        print(f"split {split}, window {label} (Hz)")
        print(feature_tests(pt, pu).round(5).to_string(index=False))
        print()

Result: reproduced. Per trial, the response frequency of HF units falls from 369 +- 100 Hz at
rest to 268 +- 131 Hz during running (150 to 250 ms; paper 309 +- 126 and 241 +- 116), both within
tolerance, with corrected p < 1e-5, while LF does not change (94 and 102 Hz, p = 0.55). The same
holds in 75 to 250 ms (370 and 279 Hz) and per unit.

## figs13-thalamic-features-predict-eeg

> On single trials, baseline thalamic rate predicts HF response frequency and LF onset
> variability, and the late EEG amplitude is predicted by ROV of HF and LF units and by HF
> response frequency, but not by LF response frequency.

Per trial at high intensity: the HF and LF features of the two claims above (window (a), k-means
split), the group's baseline rate, and the single-trial EEG late amplitude (mean over valid
channels of |EEG| in 150 to 250 ms). Models from the Methods with statsmodels mixedlm; readings
(a) raw values, (b) the code's transforms (log of the EEG amplitude and of the frequencies, every
term centred).

In [ ]:
G = grouped("k-means (a)")
feat = {}
for grp in ["HF", "LF"]:
    d = G[G.group == grp]
    gb = d.groupby(["subject", "trial"])
    feat[f"Base{grp}"] = gb.rate_base.mean()
    feat[f"ROV{grp}"] = gb.lat_late.apply(lambda x: x.std() * 1e3 if x.notna().sum() >= 2 else np.nan)
    feat[f"Freq{grp}"] = gb.isi_late.apply(lambda x: 1.0 / x.median() if x.notna().any() else np.nan)
F13 = pd.DataFrame(feat)
eeg_amp = pd.concat({r["subject"]: pd.Series(r["eeg_single_late"]) for r in S.values()}, names=["subject", "trial"])
F13["EEG"] = eeg_amp.reindex(F13.index)
F13 = F13.reset_index()


def fit13(df, formula, transform):
    d = df.copy()
    if transform:
        for c in ["EEG", "FreqHF", "FreqLF"]:
            if c in d:
                d[c] = np.log(d[c])
        for c in d.columns:
            if c not in ("subject", "trial"):
                d[c] = d[c] - d[c].mean()
    return smf.mixedlm(formula, d, groups=d["subject"]).fit()


rows = []
expect = {("ROVHF", "BaseHF"): False, ("FreqHF", "BaseHF"): True, ("ROVLF", "BaseLF"): True, ("FreqLF", "BaseLF"): False,
          ("EEG", "ROVHF"): True, ("EEG", "FreqHF"): True, ("EEG", "ROVLF"): True, ("EEG", "FreqLF"): False}
for transform in [False, True]:
    for y, x in [("ROVHF", "BaseHF"), ("FreqHF", "BaseHF"), ("ROVLF", "BaseLF"), ("FreqLF", "BaseLF")]:
        d = F13.dropna(subset=[y, x])
        f = fit13(d[["subject", "trial", y, x]], f"{y} ~ {x}", transform)
        rows.append({"reading": "(b) code transforms" if transform else "(a) raw", "model": f"{y} ~ {x}", "term": x, "trials": len(d),
                     "mice": d.subject.nunique(), "coef": f.params[x], "p": f.pvalues[x], "paper_significant": expect[(y, x)]})
    d = F13.dropna(subset=["EEG", "ROVHF", "FreqHF", "ROVLF", "FreqLF"])
    if len(d) > 10 and d.subject.nunique() > 1:
        f = fit13(d[["subject", "trial", "EEG", "FreqHF", "ROVHF", "FreqLF", "ROVLF"]], "EEG ~ FreqHF + ROVHF + FreqLF + ROVLF", transform)
        for x in ["ROVHF", "FreqHF", "ROVLF", "FreqLF"]:
            rows.append({"reading": "(b) code transforms" if transform else "(a) raw", "model": "EEG ~ FreqHF + ROVHF + FreqLF + ROVLF", "term": x,
                         "trials": len(d), "mice": d.subject.nunique(), "coef": f.params[x], "p": f.pvalues[x], "paper_significant": expect[("EEG", x)]})
    else:
        print(f"EEG model not fitted: {len(d)} trials with all four features from {d.subject.nunique()} mice")
r13 = pd.DataFrame(rows)
r13["matches_paper"] = (r13.p < 0.05) == r13.paper_significant
print(r13.round(5).to_string(index=False))
print("\nTrials with each feature:", F13[["ROVHF", "FreqHF", "ROVLF", "FreqLF", "EEG"]].notna().sum().to_dict())

Result: not reproduced. Five of the eight effects match the paper's pattern under both
readings: baseline rate predicts HF frequency (p < 1e-5) and not HF ROV (p = 0.085); the EEG
amplitude is predicted by HF frequency and LF ROV (p < 1e-5) and not LF frequency (p = 0.58 and
0.30). Baseline rate does not predict LF ROV (p = 0.65) and does predict LF frequency (p = 0.013
and 0.0035), and HF ROV does not predict the EEG (p = 0.96 and 0.92). The LF features rest on 4
mice and the EEG model on 262 trials from 3 mice.

## figs13d-lf-crosscorrelation

> Cross-correlation between LF thalamic units differs between rest and running, while that
> between HF units does not.

Per pair of units of the same group in a session (k-means split), in the late window: 2 ms bins of
the absolute delays between their spikes on the same trial, divided by the number of seed spikes;
the value is the mean of the first five bins (Methods). Rest and active per pair, signed-rank.
Window (a) 150 to 250 ms or (b) 75 to 250 ms.

In [ ]:
grp = hf_lf_groups("k-means (a)")
rows = []
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    ok = ((t.intensity == "high") & t.is_valid).to_numpy()
    uid = r["units"].unit_id
    members = {}
    for i, (rel, trial) in r["spikes"].items():
        key = (r["subject"], int(uid.loc[i]))
        if key in grp.index:
            members.setdefault(grp.loc[key], []).append((rel, trial))
    for g, us in members.items():
        for wl, win in [("(a)", russo.LATE), ("(b)", russo.REBOUND)]:
            per = []
            for rel, trial in us:
                m = (rel >= win[0]) & (rel < win[1])
                per.append([rel[m & (trial == k)] for k in range(len(t))])
            for a in range(len(us)):
                for b in range(a + 1, len(us)):
                    v = {}
                    for st in ["rest", "active"]:
                        sel = ok & (t.active == (st == "active")).to_numpy()
                        idx = np.nonzero(sel)[0]
                        v[st] = russo.crosscorr_value([per[a][k] for k in idx], [per[b][k] for k in idx])
                    rows.append({"subject": r["subject"], "group": g, "window": wl, **v})
cc = pd.DataFrame(rows).dropna()
out = []
for (g, wl), d in cc.groupby(["group", "window"]):
    w = stats.wilcoxon(d.rest, d.active) if len(d) > 1 else None
    out.append({"group": g, "window": wl, "pairs": len(d), "mice": d.subject.nunique(), "rest_median": d.rest.median(),
                "active_median": d.active.median(), "rest_larger": int((d.rest > d.active).sum()), "p": w.pvalue if w else np.nan})
print(pd.DataFrame(out).round(5).to_string(index=False))

Result: not reproduced. LF pairs (48 pairs, 4 mice) differ between states (p = 0.0065), but
they are more correlated during running (rest larger in 16 of 48), the reverse of the criterion's
direction, and HF pairs (9588 pairs) also differ (p < 1e-5) in both windows.

## discussion-orthodromic-initial-excitation

> The first-spike latency of the initial response is more variable in SM-TH than in MOs, and
> does not differ between SM-TH nuclei and TRN.

Per responsive unit at high intensity, the median and SD over trials of the first-spike latency
in 3 to 50 ms. SM-TH are the relay units, MOs all responsive MOs units (RS and FS), TRN the
thalamic units under 0.35 ms. Readings (criterion): (a) rank-sum on unit values; (b) signed-rank
on per-mouse medians for mice with units in both groups.

In [ ]:
d = H.copy()
d["grp"] = d.cls.replace({"relay": "SM-TH", "RS": "MOs", "FS": "MOs"})
pu = d.groupby(["grp", "subject", "unit"]).lat_early.agg(["median", "std"]).reset_index()
rows = []
for meas, g1, g2 in [("std", "SM-TH", "MOs"), ("median", "SM-TH", "TRN")]:
    a, b = pu[pu.grp == g1][meas].dropna(), pu[pu.grp == g2][meas].dropna()
    pm = pu.groupby(["subject", "grp"])[meas].median().unstack()[[g1, g2]].dropna()
    rows.append({"measure": "latency SD" if meas == "std" else "latency", "groups": f"{g1} vs {g2}", "units": f"{len(a)} / {len(b)}",
                 "median_1_ms": a.median() * 1e3, "median_2_ms": b.median() * 1e3, "p_ranksum": stats.mannwhitneyu(a, b).pvalue,
                 "mice": len(pm), "p_signedrank_mice": stats.wilcoxon(pm[g1], pm[g2]).pvalue if len(pm) > 1 else np.nan,
                 "mice_1_larger": int((pm[g1] > pm[g2]).sum())})
print(pd.DataFrame(rows).round(5).to_string(index=False))

In [ ]:
# Variant: TRN also counts the units under 0.35 ms located in RT itself (TRN-RT)
d = H.copy()
d["grp"] = d.cls.replace({"relay": "SM-TH", "RS": "MOs", "FS": "MOs", "TRN-RT": "TRN"})
pu = d.groupby(["grp", "subject", "unit"]).lat_early.agg(["median", "std"]).reset_index()
a, b = pu[pu.grp == "SM-TH"]["median"].dropna(), pu[pu.grp == "TRN"]["median"].dropna()
pm = pu.groupby(["subject", "grp"])["median"].median().unstack()[["SM-TH", "TRN"]].dropna()
print(f"Latency, SM-TH relay against TRN with RT units: {len(a)} / {len(b)} units, medians {a.median() * 1e3:.2f} and "
      f"{b.median() * 1e3:.2f} ms, rank-sum p = {stats.mannwhitneyu(a, b).pvalue:.4f}; {len(pm)} mice, "
      f"signed-rank p = {stats.wilcoxon(pm['SM-TH'], pm['TRN']).pvalue:.4f}")

Result: reproduced. The first-spike latency in 3 to 50 ms is more variable in SM-TH relay
units than in MOs units (median SD 2.55 against 1.28 ms, rank-sum p < 1e-5; higher in 7 of 7 mice,
p = 0.016), and the latency of SM-TH relay and TRN units does not differ (8.10 against 10.40 ms,
p = 0.073; per mouse p = 0.44). With the RT unit added to TRN the rank-sum on units gives p =
0.0496 (8.10 against 10.24 ms) and the per-mouse test p = 0.44, so the claim holds under reading
(b) with either TRN class.

## source-data-link

> The dandiset holds the mouse sessions behind the paper's Figure 1C and Figure 2B, so that values
> computed from it agree with the per-session and per-subject values in the paper's Source Data
> file.

No identifier links the Source Data rows to sessions, so the distributions are compared
(criterion): the session count of Figure 1C (26 +- 3), its two medians (within 10%), and the five
Figure 2B means (at least four within 10%), computed with the methods of fig1c and fig2b. The
Source Data file is read from the publisher's site and checked against its SHA-256.

In [ ]:
SOURCE_DATA_URL = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-58717-2/"
                   "MediaObjects/41467_2025_58717_MOESM4_ESM.xlsx")
SOURCE_DATA_SHA256 = "5feebc5a6c6ced1441d0a74af35c7b18e1c2064f1e633f88c74ab0cd8689d0f7"
raw = requests.get(SOURCE_DATA_URL, timeout=120).content
print("Source Data SHA-256 matches:", hashlib.sha256(raw).hexdigest() == SOURCE_DATA_SHA256, f"({len(raw)} bytes)")
xl = pd.ExcelFile(io.BytesIO(raw))


def paired_sheet(name):
    """Sheets with one value per subject: the two value columns below their header row."""
    d = pd.read_excel(xl, sheet_name=name, header=None)
    hdr = d.index[d.iloc[:, 1:].notna().sum(1) >= 2][0]
    cols = [c for c in d.columns[1:] if pd.notna(d.loc[hdr, c])]
    out = d.loc[hdr + 1:, cols].astype(float).dropna(how="all")
    out.columns = [str(d.loc[hdr, c]).strip() for c in cols]
    return out.reset_index(drop=True)


def box_sheet(name):
    """Box-plot sheets: for each block (title in row 1), rows are states and columns Min .. Max."""
    d = pd.read_excel(xl, sheet_name=name, header=None)
    out = {}
    titles = [(c, str(d.iloc[1, c])) for c in d.columns if pd.notna(d.iloc[1, c])]
    for c, title in titles:
        lab_col = c - 1
        rows = {}
        for i in range(5, len(d)):
            lab = d.iloc[i, lab_col]
            if pd.notna(lab):
                rows[str(lab).strip()] = d.iloc[i, c:c + 5].astype(float).to_numpy()
        out[title] = pd.DataFrame(rows, index=["min", "q25", "median", "q75", "max"]).T
    return out


sd1c = paired_sheet("Figure 1 C")
sd2b = pd.read_excel(xl, sheet_name="Figure 2 B", header=2).iloc[:, 1:]
print("Figure 1 C: %d pairs, medians rest %.1f, active %.1f uV" % (len(sd1c), sd1c.iloc[:, 0].median(), sd1c.iloc[:, 1].median()))
print("Figure 2 B: per-subject latencies, n and mean (ms):", {c: (int(sd2b[c].notna().sum()), round(sd2b[c].mean() * 1e3, 1)) for c in sd2b.columns})

rows = []
for label, (intens, chan) in READINGS.items():
    t = state_table("eeg", intens, chan=chan).dropna(subset=["rest", "active"])
    rows.append({"reading": label, "sessions": len(t), "rest_median": t.rest.median(), "active_median": t.active.median(),
                 "rest_vs_SD_%": 100 * (t.rest.median() / sd1c.iloc[:, 0].median() - 1),
                 "active_vs_SD_%": 100 * (t.active.median() / sd1c.iloc[:, 1].median() - 1)})
print("\nFigure 1C part:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

# Figure 2B part: per mouse, mean latency over all high-intensity valid trials (both states)
per_mouse = {}
for r in S.values():
    if not r.get("has_probes"):
        continue
    t = r["trials"]
    hi = ((t.intensity == "high") & t.is_valid).to_numpy()
    vals = {"EEG (s)": np.nanmean(r["eeg_lat"][hi]), "LFP (s)": np.nanmean(r["lfp_lat"][hi]), "CSD (s)": np.nanmean(r["csd_lat"][hi])}
    for name, cls in [("SM-TH (s)", "relay"), ("MOs (s)", "RS")]:
        s = trial_unit_latency(cls)
        if r["subject"] in s.index.get_level_values(0):
            v = s.loc[r["subject"]].reindex(range(len(t)))[hi]
            vals[name] = v.mean()
    per_mouse[r["subject"]] = vals
pm2b = pd.DataFrame(per_mouse).T
cmp = pd.DataFrame({"mice": pm2b.notna().sum(), "this_mean_ms": pm2b.mean() * 1e3, "SD_n": sd2b.notna().sum(),
                    "SD_mean_ms": sd2b.mean() * 1e3})
cmp["diff_%"] = 100 * (cmp.this_mean_ms / cmp.SD_mean_ms - 1)
print("\nFigure 2B part:")
print(cmp.round(2).to_string())
print("Within 10%:", int((cmp["diff_%"].abs() <= 10).sum()), "of 5")

Result: partially reproduced. The Figure 2B part holds: the five mean latencies are within 10%
of the Source Data means (-2.0 to +9.5%). The Figure 1C part does not: the dandiset holds 10 deep
MOs sessions (20 with high and medium), against 26 in the Source Data, and the medians are 16.5%
(rest) and 37.2% (active) below the Source Data medians at high intensity.

## Human claims on the Source Data file

The human recordings are not public. The claims below are tested, as their criteria say, on the
per-subject values or box-plot summaries in the paper's Source Data file (`data: elsewhere`).

### fig1a-tms-late-rest-active

> In 12 healthy subjects, the late (150 to 250 ms) TMS-evoked EEG response over premotor channels
> has a larger rectified amplitude and phase locking at rest than during active hand squeezing
> (amplitude 1.37 +- 0.59 against 0.77 +- 0.22 uV; PLF 0.21 +- 0.07 against 0.15 +- 0.05).

In [ ]:
a1 = paired_sheet("Figure 1 A")
rest, act = a1.iloc[:, 0], a1.iloc[:, 1]
w = stats.wilcoxon(rest, act)
print(f"Figure 1 A: {len(a1)} subjects; rest {rest.mean():.3f} +- {rest.std():.3f} uV, active {act.mean():.3f} +- {act.std():.3f} uV; "
      f"rest larger in {int((rest > act).sum())}; two-sided signed-rank p = {w.pvalue:.4g} (paper 0.002)")
print(f"Within tolerance: rest |{rest.mean():.3f} - 1.37| <= 0.59: {abs(rest.mean() - 1.37) <= 0.59}; "
      f"active |{act.mean():.3f} - 0.77| <= 0.22: {abs(act.mean() - 0.77) <= 0.22}")
s1a = box_sheet("Figure S1A")
for k, v in s1a.items():
    print(f"\nFigure S1A, {k}:")
    print(v.to_string())

Result: reproduced. The 12 Source Data pairs give 1.379 +- 0.588 uV at rest and 0.771 +- 0.223
uV active, within the paper's values, rest larger in 12 of 12 (two-sided signed-rank p = 0.00049;
paper 0.002). The PLF medians are 0.215 at rest and 0.165 active; the PLF test cannot be rerun
from the box-plot summary.

### fig1b-ieeg-late-rest-active

> In 5 epilepsy patients, the late (150 to 250 ms) iEEG response to single-pulse electrical
> stimulation of premotor cortex is smaller in rectified amplitude and PLF during active hand
> squeezing than at rest.

In [ ]:
b1 = paired_sheet("Figure 1 B")
rest, act = b1.iloc[:, 0], b1.iloc[:, 1]
print(f"Figure 1 B: {len(b1)} subjects; rest mean {rest.mean():.3f}, active {act.mean():.3f} (z); "
      f"active lower in {int((act < rest).sum())} of {len(b1)}; differences {np.round((rest - act).to_numpy(), 3).tolist()}")
print(f"Signed-rank on the 5 pairs (not the paper's test): p = {stats.wilcoxon(rest, act).pvalue:.4f}")
iplf = [v for k, v in s1a.items() if "SPES" in k][0]
print(f"Figure S1A iEEG PLF medians: rest {iplf.loc['Rest', 'median']}, active {iplf.loc['Active', 'median']}")

Result: reproduced. Active is lower than rest in 5 of 5 subjects (means 5.28 and 3.95 z), and
the iEEG PLF medians are 0.515 at rest and 0.44 active. The paper's mixed model on contacts cannot
be rerun from the public values.

### figs2-tms-controls

> The TMS late-component modulation is also seen with passive hand movement and with leg cycling,
> is absent for posterior TMS, is absent for median nerve somatosensory evoked potentials, and
> does not differ between hand closing and opening.

In [ ]:
g = paired_sheet("Figure S2G")
w = stats.wilcoxon(g.iloc[:, 0], g.iloc[:, 1])
print(f"Figure S2G: {len(g)} subjects, columns {list(g.columns)}; means {g.mean().round(3).to_dict()}; "
      f"two-sided signed-rank p = {w.pvalue:.4f} (paper 0.57)")
late = {}
for sheet in ["Figure S2A", "Figure S2C", "Figure S2D", "Figure S2E"]:
    b = box_sheet(sheet)
    k = [t for t in b if t.lower().startswith("late rect")][0]
    late[sheet] = b[k]["median"]
    print(f"{sheet}, late rectified amplitude medians: {b[k]['median'].to_dict()}")
print("S2A passive median below rest:", late["Figure S2A"]["Passive"] < late["Figure S2A"]["Rest"],
      "; S2C cycling median below rest:", late["Figure S2C"]["Cycling"] < late["Figure S2C"]["Rest"])

Result: reproduced, as far as the public data allow. Contraction and relaxation do not differ
(12 pairs, p = 0.61; paper 0.57), and the late amplitude median at rest (1.2 and 0.8 uV) exceeds
the passive (0.65) and cycling (0.62) medians. The absence of modulation for posterior TMS and
SSEP is not testable from the box-plot summaries (medians rest 0.75, active 0.6; and 0.75, 0.8).

### figs2a-tms-early-component

> The early (3 to 50 ms) TMS-evoked component is reduced in amplitude during active movement but
> not in phase locking (2.24 +- 0.45 against 1.88 +- 0.32 uV; PLF 0.43 in both), and its
> reduction is not correlated with that of the late component.

In [ ]:
b = box_sheet("Figure S2A")
early = b[[t for t in b if t.lower().startswith("early rect")][0]]
eplf = b[[t for t in b if t.lower().startswith("early plf")][0]]
print("Early rectified amplitude (uV):"); print(early.to_string())
print("Early PLF:"); print(eplf.to_string())
r_, a_ = early.loc["Rest", "median"], early.loc["Active", "median"]
print(f"rest median {r_} > active median {a_}: {r_ > a_}; rest within 2.24 +- 0.45: {abs(r_ - 2.24) <= 0.45}; "
      f"active within 1.88 +- 0.32: {abs(a_ - 1.88) <= 0.32}")

Result: reproduced, as far as the public data allow. The early amplitude median is 2.15 uV at
rest and 1.75 uV active, each within the paper's mean and SD; the early PLF medians are 0.41 and
0.40. The tests and the correlation cannot be recomputed from box-plot summaries.

### figs3b-ieeg-early-no-modulation

> The early (3 to 50 ms) iEEG response to electrical stimulation in humans is not modulated by
> state, in monopolar or bipolar derivation.

In [ ]:
d = pd.read_excel(xl, sheet_name="Figure S3B", header=None)
rows = []
for c in range(1, d.shape[1]):
    title = d.iloc[1, c]
    if pd.isna(title):
        continue
    v = d.iloc[5:, [c, c + 1]].astype(float).dropna()
    w = stats.wilcoxon(v.iloc[:, 0], v.iloc[:, 1])
    rows.append({"measure": str(title), "subjects": len(v), "rest_mean": v.iloc[:, 0].mean(), "active_mean": v.iloc[:, 1].mean(),
                 "differences_rest_minus_active": np.round((v.iloc[:, 0] - v.iloc[:, 1]).to_numpy(), 3).tolist(), "p_signedrank": w.pvalue})
print(pd.DataFrame(rows).to_string(index=False))
print("With five pairs the smallest two-sided exact signed-rank p is 0.0625.")

Result: reproduced, on a test that cannot find an effect. None of the four signed-rank tests on
the five subjects gives p < 0.05 (p = 0.31 to 0.88), and the per-subject differences are small
and of mixed sign (-0.08 to +0.09). With five pairs the smallest possible p is 0.0625.

## Model claims (fig4e, fig4f, fig4hi, figs14a, figs14b)

The criteria ask for a rerun of the paper's model, the `stim` branch of
github.com/bazhlab-ucsd/sleep-stage-transition (commit 76ea885), with the analysis code on figshare
(doi:10.6084/m9.figshare.25773501.v1, MATLAB). The Methods (Thalamocortical network model) give
the two states: rest as the parameters were tuned, active with the CX to TC and CX to RE
miniEPSPs raised from 30 to 50 and the thalamic acetylcholine level lowered from 1.3 to 1; 30
trials of 15 s (15 rest, 15 active), the pulse at 8 to 9.75 s in steps of 125 ms. The cell below
fetches the model at the commit, builds it in a temporary directory with the Makefile, and times
200 ms of simulation of the published network, to estimate what the 30 trials cost on this
runner (4 CPUs).

In [ ]:
import os
import subprocess
import tarfile
import tempfile

MODEL_REPO = "bazhlab-ucsd/sleep-stage-transition"   # branch stim, the paper's model (claims file)
MODEL_COMMIT = "76ea8853fa4e97ac8c47032bc00f378a0920f9d0"
TEST_MS = 200                                         # simulated time of each timing run


def set_params(text, **kw):
    lines = [(f"{k.split()[0]} {kw[k.split()[0]]}" if k.strip() and k.split()[0] in kw else k) for k in text.splitlines()]
    return "\n".join(lines) + "\n"


with tempfile.TemporaryDirectory() as tmp:
    tgz = requests.get(f"https://codeload.github.com/{MODEL_REPO}/tar.gz/{MODEL_COMMIT}", timeout=300).content
    tarfile.open(fileobj=io.BytesIO(tgz)).extractall(tmp, filter="data")
    src = os.path.join(tmp, os.listdir(tmp)[0])
    params = open(os.path.join(src, "params.txt")).read()
    keys = ["tmax", "num_mp_threads", "stim_cx_start", "stim_cx_end", "stim_cx_strength", "stim_cx_start_neuron",
            "stim_cx_end_neuron", "stim_in_start_neuron", "stim_in_end_neuron", "ach_cx_awake", "ach_th_awake"]
    print(f"{MODEL_REPO} at {MODEL_COMMIT[:7]}: the parameter set in params.txt")
    print({l.split()[0]: l.split()[1] for l in params.splitlines() if l.strip() and l.split()[0] in keys})
    print("CX to TC and CX to RE AMPA_D2 lines of network.cfg (Mini_s is the miniEPSP strength):")
    for l in open(os.path.join(src, "network.cfg")).read().splitlines():
        if l.split()[:3] in (["CX", "TC", "AMPA_D2"], ["CX", "RE", "AMPA_D2"]):
            print("  " + l.strip())
    t0 = time.time()
    for cmd in (["make", "-s", "mindcurrent"], ["make", "-s", "network"]):
        r = subprocess.run(cmd, cwd=src, capture_output=True, text=True)
        print(" ".join(cmd), "exit", r.returncode)
    with open(os.path.join(src, "connection_info2"), "w") as fh:
        subprocess.run(["./generate_network", "network.cfg"], cwd=src, stdout=fh, stderr=subprocess.DEVNULL, check=True)
    print(f"built the model and its network in {time.time() - t0:.0f} s")
    speed = {}
    for threads in [1, 4]:
        os.makedirs(os.path.join(src, f"run{threads}"))
        with open(os.path.join(src, f"run{threads}", "params.txt"), "w") as fh:
            fh.write(set_params(params, tmax=TEST_MS, num_mp_threads=threads))
        t0 = time.time()
        subprocess.run(["./mindcurrent", f"run{threads}/params.txt", f"run{threads}", "connection_info2"], cwd=src,
                       capture_output=True, check=True)
        speed[threads] = (time.time() - t0) / (TEST_MS / 1000)
        print(f"{threads} thread(s): {TEST_MS} ms simulated in {time.time() - t0:.1f} s, "
              f"{speed[threads]:.0f} s per simulated second")

# Methods: 30 trials (15 rest, 15 active), pulse at 8 to 9.75 s in steps of 125 ms, trials of 15 s
pulses = 8.0 + 0.125 * np.arange(15)
sim_min = 2 * (pulses + 0.25).sum()        # each trial simulated only to 250 ms after its pulse
sim_full = 30 * 15.0
for label, s in [("to 250 ms after each pulse", sim_min), ("15 s trials (Methods)", sim_full)]:
    print(f"30 trials {label}: {s:.0f} s simulated; four single-thread runs at once "
          f"{s * speed[1] / 4 / 3600:.1f} h, one four-thread run {s * speed[4] / 3600:.1f} h")
print("Notebook limit 1 h (docs/protocol.md); fig4hi-model-gaba-contributions adds runs with TRN silenced, "
      "IN silenced and a grid of GABA-A strengths")

Blocked (`not_runnable`). The model builds and runs: one parameter set ships with the branch
(the rest state: thalamic acetylcholine 1.3, CX to TC and CX to RE miniEPSP strength 30), and the
active state is a change of three values the Methods give. The obstacle is time: on this runner
one simulated second takes about 120 s on one thread and 80 s on four (cell above; the
timing varies between runs: 121 to 123 s and 82 to 83 s in the three runs of the last revision, 158 s
and 113 s in an earlier run). The 30 trials of fig4e,
fig4f, figs14a and figs14b need 274 s of simulation even when each trial stops 250 ms after its
pulse (450 s for the Methods' 15 s trials), about 2.3 h with four single-thread runs at once (3.8 h for 15 s trials),
against the notebook's one-hour limit; fig4hi adds runs with TRN silenced, with IN silenced and
over a grid of GABA-A strengths. The analysis code is MATLAB and was not ported.

## Summary

| Claim | Outcome |
|---|---|
| source-data-link | partially reproduced |
| fig1a-tms-late-rest-active | reproduced (Source Data) |
| fig1b-ieeg-late-rest-active | reproduced (Source Data) |
| figs2-tms-controls | reproduced (Source Data) |
| figs2a-tms-early-component | reproduced (Source Data) |
| figs3b-ieeg-early-no-modulation | reproduced (Source Data) |
| fig5a-human-mouse-similar | blocked, data missing |
| fig1c-eeg-late-rest-larger | reproduced |
| fig1d-lfp-late-rest-larger | reproduced |
| fig1e-csd-late-rest-larger | reproduced |
| figs1b-mouse-plf | reproduced |
| figs4-intensity-dependence | reproduced |
| speed-threshold-robust | not reproduced |
| figs5-early-component-mouse | reproduced |
| figs6-posterior-no-modulation | blocked, data missing |
| triphasic-pattern | partially reproduced |
| fig2b-thalamus-leads | not reproduced |
| figs7a-active-shorter-latency | reproduced |
| figs7b-latency-correlation | reproduced |
| fig3a-thalamic-rate-higher-active | reproduced |
| figs9-mos-rate-no-modulation | not reproduced |
| figs10a-fs-trn-no-modulation | reproduced (fragile) |
| fig3b-bimodal-isi | not reproduced |
| figs11d-hf-earlier-less-variable | not reproduced |
| figs12-intensity-shifts-to-hf | reproduced |
| fig3e-baseline-rate-doubles | reproduced |
| fig3e-lf-synchrony-reduced | not reproduced |
| fig3e-hf-frequency-reduced | reproduced |
| figs13-thalamic-features-predict-eeg | not reproduced |
| figs13d-lf-crosscorrelation | not reproduced |
| discussion-orthodromic-initial-excitation | reproduced |
| optogenetic claims (3) | second notebook |
| model claims (5) | blocked, not runnable (too slow for the time limit) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset holds 10 of the paper's mouse
sessions with deep MOs stimulation (the paper: 26 EEG sessions from 15 mice, units from 21 mice),
so every mouse test has fewer animals. Bad channels and trials were removed in the paper by
visual inspection and a lab spreadsheet; here all channels and trials, and the file's own
`is_data_valid` and `is_valid` flags, are both shown. Responsive units use 10 ms bins in place of
the paper's adaptive kernel. The dip test uses the `diptest` package. The CSD averages the two
sites of each Neuropixels row. The rest against running rate tests of fig3a and figs10a
are shown also within sessions (van Elteren), as pooling trials over sessions mixes sessions
with different rates and fractions of active trials. The HF and LF groups are a k-means split
with 18 LF units (8 at the paper's 17.88 ms), so every HF and LF claim is tested on groups unlike
the paper's 569 HF and 700 LF units. The model claims were built and timed, not run. The human claims use the per-subject values and box-plot summaries
of the Source Data file, as their criteria say.

What was found about the dataset. The units' `waveform_duration` is described as seconds and
holds milliseconds. The trial table's `is_running` uses a 0 cm/s threshold, not the paper's 0.1
cm/s. No session has stimulation of the posterior control areas of Figure S6. The EEG, LFP, units
and trial table share a clock: in the trial-averaged raw signals the stimulation artifact sits within
0.2 ms of the trial onset in the EEG and within 0.6 ms (1.5 samples) in the LFP of every session, and units fire in 2 to 10 ms after the pulse in every mouse. The relay units' evoked ISIs
are unimodal here (median 2.6 ms), with few units above the paper's 17.88 ms threshold, so the HF
and LF split of the paper's Figure 3 cannot be recovered from this subset.

In [ ]:
print(f"Total run time {(time.time() - started) / 60:.1f} min")